# BOLD Mortality Prediction: First Pass

Predicting whether an ICU patient died during their hospital admission, using only labs
and vitals available at a single point in time, and explaining *which* markers drive
that prediction rather than just producing a score.

Three models compared on the same pipeline: Logistic Regression as the simple
baseline, Random Forest and Gradient Boosting as the more capable options. SHAP and LR
coefficients explain what each model relies on, then a confounder check tests whether
the top markers hold up once age and severity are accounted for.


## Step 0: Data access

Real BOLD is a credentialed PhysioNet dataset and certification is still in progress. Until
`bold_dataset.csv` is in this folder, the notebook runs on a synthetic stand-in with the same
column names. That is enough to build the pipeline, not enough to trust any result.

The cell below picks up whichever file is present, so switching over means dropping
the real file into this folder and rerunning. Nothing to edit.

In [ ]:
from pathlib import Path

# One constant decides which dataset the whole notebook runs on. Real BOLD if it has
# been downloaded, the synthetic stand-in otherwise, so this still runs while
# PhysioNet access is pending.
REAL_DATA = Path("bold_dataset.csv")
SYNTHETIC_DATA = Path("synthetic_bold_dataset.csv")

IS_SYNTHETIC = not REAL_DATA.exists()
DATA_PATH = SYNTHETIC_DATA if IS_SYNTHETIC else REAL_DATA

# Loud on purpose. Synthetic numbers look exactly like real ones in the output.
if IS_SYNTHETIC:
    print("=" * 72)
    print("SYNTHETIC DATA, NOT A REAL FINDING")
    print("Fabricated values under real BOLD column names. Good enough to prove the")
    print("pipeline runs, worthless as a result. Do not quote any number below.")
    print("=" * 72)
else:
    print("Real BOLD data.")

print(f"DATA_PATH: {DATA_PATH}")

## Step 1: Load and look

Look at the data before touching it. Check how many patients, how many columns, and how much of each column is filled in.

Some columns are mostly full because those tests get run on nearly every patient. Others have more gaps because they only get ordered when a doctor already suspects 
something specific. That's expected, not a problem.

Nothing gets dropped or changed yet. This step is just looking.

In [2]:
import pandas as pd

# Don't truncate long tables.
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 120)

df = pd.read_csv(DATA_PATH)

print(f"Loaded:   {DATA_PATH}")
print(f"Patients: {df.shape[0]:,}")
print(f"Columns:  {df.shape[1]}")

Loaded:   synthetic_bold_dataset.csv
Patients: 2,000
Columns:  71


In [3]:
# Count columns by type, then list the non-numeric ones.
print("Column types:")
print(df.dtypes.value_counts())
print()

text_cols = df.select_dtypes(exclude="number").columns.tolist()
print("Text columns:", ", ".join(text_cols) if text_cols else "none")

Column types:
float64    51
int64      18
str         2
Name: count, dtype: int64

Text columns: source_db, race_ethnicity


In [4]:
# Missing values per column, worst first.
missing = pd.DataFrame({
    "n_missing": df.isna().sum(),
    "pct_missing": (df.isna().mean() * 100).round(1),
}).sort_values("pct_missing", ascending=False)

print("Missing data per column (worst first):")
print(missing.to_string())

Missing data per column (worst first):
                                 n_missing  pct_missing
others_ck_mb                          1109         55.4
others_ck_cpk                          902         45.1
hfp_bilirubin_direct                   806         40.3
others_ld_ldh                          780         39.0
coag_fibrinogen                        687         34.4
hfp_albumin                            669         33.4
hfp_alp                                653         32.6
hfp_bilirubin_total                    615         30.8
hfp_alt                                611         30.6
hfp_ast                                594         29.7
bmp_lactate                            476         23.8
coag_ptt                               413         20.6
coag_inr                               293         14.6
coag_pt                                286         14.3
cbc_rdw                                198          9.9
bmp_bun                                  0          0.0
cbc_rbc  

In [5]:
# Bucket columns by how much is missing.
pct = missing["pct_missing"]
buckets = {
    "Complete (0%)":       (pct == 0).sum(),
    "Usable (1-30%)":      ((pct > 0) & (pct <= 30)).sum(),
    "Patchy (31-60%)":     ((pct > 30) & (pct <= 60)).sum(),
    "Mostly empty (>60%)": (pct > 60).sum(),
}
for name, n in buckets.items():
    print(f"{name:<22}{n:>3} columns")
print()

Complete (0%)          56 columns
Usable (1-30%)          6 columns
Patchy (31-60%)         9 columns
Mostly empty (>60%)     0 columns



## Step 2: Define the label

This is what we're actually predicting: `in_hospital_mortality`. A value of 1 means the patient died, 0 means they survived. BOLD already provides this label directly, nothing to build.

The important part is how rare deaths are, because that changes everything else we do later:

1. **Accuracy won't tell us anything useful.** A model that just guesses "survived" every time would still be right about 85% of the time, while catching zero deaths. That's why we use AUROC and AUPRC instead (Step 5), not plain accuracy.
2. **The model needs to be told deaths matter more.** Otherwise it will happily ignore the rare cases to keep its overall error low. 
3. **The train/test split has to keep the same death rate in both halves.** Otherwise one half could end up with way more or fewer deaths than the other by chance (Step 4).

In [6]:
LABEL = "in_hospital_mortality"

# Fail early if the label has gaps or isn't 0/1.
assert LABEL in df.columns, f"'{LABEL}' not found"
assert df[LABEL].isna().sum() == 0, "Label has missing values"
assert set(df[LABEL].unique()) <= {0, 1}, f"Expected 0/1, got {sorted(df[LABEL].unique())}"

print(f"Label: {LABEL}")
print("No gaps, values are 0/1 only.")

Label: in_hospital_mortality
No gaps, values are 0/1 only.


In [7]:
# Count deaths against survivors.
counts = df[LABEL].value_counts()
n_survived = int(counts.get(0, 0))
n_died = int(counts.get(1, 0))
pct_died = n_died / len(df) * 100

print(f"Survived: {n_survived:>6,}  ({100 - pct_died:.1f}%)")
print(f"Died:     {n_died:>6,}  ({pct_died:.1f}%)")
print(f"Total:    {len(df):>6,}")
print()
print(f"Roughly {n_survived / n_died:.1f} survivors for every death.")

Survived:  1,695  (84.8%)
Died:        305  (15.2%)
Total:     2,000

Roughly 5.6 survivors for every death.


In [8]:
# Sanity checks on the death rate.
print(f"An 'everyone survives' model would score {100 - pct_died:.1f}% accuracy.")
print("Which is exactly why we don't use accuracy.")
print()

if 10 <= pct_died <= 25:
    print(f"{pct_died:.1f}% mortality is plausible for ICU data.")
else:
    print(f"Note: {pct_died:.1f}% sits outside the 15-18% BOLD reports.")

An 'everyone survives' model would score 84.8% accuracy.
Which is exactly why we don't use accuracy.

15.2% mortality is plausible for ICU data.


## Step 3: Pick the predictors

For every column, ask one question: would a clinician know this at the bedside, at the
moment of the reading?

Yes, keep it (labs, vitals, blood gas, past SOFA scores, demographics). No, because
it's only known after the outcome, exclude it (future SOFA scores, length of stay,
timestamps, IDs, the outcome itself). Race/ethnicity is set aside too, not used as a
predictor, kept for a fairness check later.

Every column gets checked, nothing slips through unlabelled, and a final check
confirms nothing leaky made it into the predictor list.

In [9]:
# Match predictor groups by column prefix.

# Full blood count, clotting, metabolic panel, liver, misc enzymes.
lab_cols = [c for c in df.columns
            if c.startswith(("cbc_", "coag_", "bmp_", "hfp_", "others_"))]

# Bedside observations, plus the two oxygen saturation readings.
vital_cols = [c for c in df.columns if c.startswith("vitals_")]
vital_cols += [c for c in ["SpO2", "SaO2"] if c in df.columns]

gas_cols = [c for c in ["pH", "pCO2", "pO2", "Carboxyhemoglobin", "Methemoglobin"]
            if c in df.columns]

# Severity over the previous 24 hours.
severity_cols = [c for c in df.columns if c.startswith("sofa_past_")]

# Demographics and body measurements, all taken at admission.
demo_cols = [c for c in ["admission_age", "sex_female", "comorbidity_score_value",
                         "weight_admission", "height_admission", "BMI_admission"]
             if c in df.columns]

for name, cols in [("Labs", lab_cols), ("Vitals", vital_cols), ("Blood gas", gas_cols),
                   ("Severity (SOFA)", severity_cols), ("Demographics", demo_cols)]:
    print(f"{name} ({len(cols)}):")
    print(f"  {', '.join(cols)}")
    print()

Labs (32):
  cbc_wbc, cbc_hemoglobin, cbc_hematocrit, cbc_platelet, cbc_mch, cbc_mchc, cbc_mcv, cbc_rbc, cbc_rdw, coag_fibrinogen, coag_pt, coag_inr, coag_ptt, bmp_sodium, bmp_potassium, bmp_chloride, bmp_bicarbonate, bmp_bun, bmp_creatinine, bmp_glucose, bmp_aniongap, bmp_calcium, bmp_lactate, hfp_alt, hfp_alp, hfp_ast, hfp_bilirubin_total, hfp_bilirubin_direct, hfp_albumin, others_ck_cpk, others_ck_mb, others_ld_ldh

Vitals (8):
  vitals_heart_rate, vitals_resp_rate, vitals_mbp_ni, vitals_sbp_ni, vitals_dbp_ni, vitals_tempc, SpO2, SaO2

Blood gas (5):
  pH, pCO2, pO2, Carboxyhemoglobin, Methemoglobin

Severity (SOFA) (6):
  sofa_past_coagulation_24hr, sofa_past_liver_24hr, sofa_past_cardiovascular_24hr, sofa_past_cns_24hr, sofa_past_renal_24hr, sofa_past_overall_24hr

Demographics (6):
  admission_age, sex_female, comorbidity_score_value, weight_admission, height_admission, BMI_admission



In [10]:
# Columns the model must not see, keyed by reason.
exclusions = {}

# Measured after the index reading.
exclusions["future SOFA scores (measured after the reading)"] = [
    c for c in df.columns if c.startswith("sofa_future_")
]

# Total LOS is only known once the admission ends.
exclusions["length of stay (only known once the admission ends)"] = [
    c for c in ["los_hospital", "los_ICU"] if c in df.columns
]

# Anything recorded at or after discharge.
exclusions["timestamps and discharge fields (at or after the outcome)"] = [
    c for c in df.columns
    if "_timestamp" in c or "datetime_" in c or "discharge" in c.lower()
]

# Identifiers carry no clinical signal.
exclusions["IDs and source database (no clinical meaning)"] = [
    c for c in ["unique_subject_id", "unique_hospital_admission_id",
                "unique_icustay_id", "source_db"] if c in df.columns
]

# The answer, plus race_ethnicity, kept aside for a later check.
exclusions["the outcome, and race_ethnicity (kept aside for a later check)"] = [
    c for c in [LABEL, "race_ethnicity"] if c in df.columns
]

for reason, cols in exclusions.items():
    if cols:
        print(f"EXCLUDED: {reason}")
        print(f"  {', '.join(cols)}")
        print()

EXCLUDED: future SOFA scores (measured after the reading)
  sofa_future_coagulation_24hr, sofa_future_liver_24hr, sofa_future_cardiovascular_24hr, sofa_future_cns_24hr, sofa_future_renal_24hr, sofa_future_overall_24hr

EXCLUDED: length of stay (only known once the admission ends)
  los_hospital, los_ICU

EXCLUDED: IDs and source database (no clinical meaning)
  unique_subject_id, unique_hospital_admission_id, unique_icustay_id, source_db

EXCLUDED: the outcome, and race_ethnicity (kept aside for a later check)
  in_hospital_mortality, race_ethnicity



In [11]:
FEATURES = lab_cols + vital_cols + gas_cols + severity_cols + demo_cols

# Check every column was either kept or excluded.
all_excluded = {c for cols in exclusions.values() for c in cols}
unaccounted = [c for c in df.columns if c not in FEATURES and c not in all_excluded]

print(f"Columns in data: {len(df.columns)}")
print(f"Kept:            {len(FEATURES)}")
print(f"Excluded:        {len(all_excluded)}")
print(f"Unaccounted:     {len(unaccounted)}")
print()

if unaccounted:
    print("WARNING, neither kept nor excluded:", ", ".join(unaccounted))
else:
    print("Every column accounted for.")

# Stop if anything leaky got through.
leaky = [c for c in FEATURES
         if any(p in c for p in ("sofa_future_", "los_", "_timestamp", "datetime_", "discharge"))]
assert not leaky, f"LEAKAGE, these must not be predictors: {leaky}"
assert LABEL not in FEATURES, "LEAKAGE, the outcome is in the feature list"
print("Leakage check passed.")

Columns in data: 71
Kept:            57
Excluded:        14
Unaccounted:     0

Every column accounted for.
Leakage check passed.


## Step 4: Split the data

Three splits, not two. Train fits the models, validation is where every tuning
decision gets made, and test stays sealed until the very end so its numbers are not
something we quietly steered towards.

The split is by patient, not by row. Real BOLD records some patients more than once,
and if the same person has rows in both train and test the model is partly being
marked on people it already studied. Splitting on `unique_subject_id` stops that. The
ID only decides which pile a row lands in, it is never handed to the model.

Deaths are also rare, so the split has to keep the death rate even across the three
sets as well as keeping patients whole. `StratifiedGroupKFold` does both at once.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

X = df[FEATURES]
y = df[LABEL]

# Group rows by patient so one person cannot end up in two splits.
groups = df["unique_subject_id"]

def split_off(idx, n_folds, seed=42):
    """Hold out 1 fold of the patients in `idx`, keeping the death rate steady.

    StratifiedGroupKFold does both jobs at once: it never splits a patient across
    groups, and it keeps the share of deaths even. Plain GroupShuffleSplit only does
    the first, and let the death rate drift by 5 points on this data.
    """
    splitter = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    keep, held = next(splitter.split(idx, y.loc[idx], groups=groups.loc[idx]))
    return idx[keep], idx[held]

# 60 / 20 / 20. A fifth comes off as test, then a quarter of the rest as validation.
train_val_idx, test_idx = split_off(df.index, 5)
train_idx, val_idx = split_off(train_val_idx, 4)

X_train, y_train = X.loc[train_idx], y.loc[train_idx]
X_val,   y_val   = X.loc[val_idx],   y.loc[val_idx]
X_test,  y_test  = X.loc[test_idx],  y.loc[test_idx]

for name, idx in [("Train", train_idx), ("Validation", val_idx), ("Test", test_idx)]:
    print(f"{name:<12}{len(idx):>7,} rows  {groups.loc[idx].nunique():>7,} patients  "
          f"{y.loc[idx].mean() * 100:>5.1f}% died")

In [ ]:
# The bug this split exists to prevent: the same patient scored in two places.
patients = {name: set(groups.loc[idx]) for name, idx in
            [("train", train_idx), ("validation", val_idx), ("test", test_idx)]}
assert not patients["train"] & patients["test"], "patient in both train and test"
assert not patients["train"] & patients["validation"], "patient in both train and validation"
assert not patients["validation"] & patients["test"], "patient in both validation and test"

# Deaths are rare, so an uneven split would make the three sets hard to compare.
rates = [y.loc[idx].mean() for idx in (train_idx, val_idx, test_idx)]
spread = max(rates) - min(rates)
assert spread < 0.02, f"death rate drifted between splits: {rates}"

print("So: no patient appears in more than one split, and the death rate varies by only")
print(f"{spread * 100:.1f} percentage points across the three, so they are comparable.")

## Step 5: Train Random Forest

Some columns still have gaps. Fill them with the median value of each column, learned from the training set only, so the test set stays something the model has never seen.

Then train a Random Forest. `min_samples_leaf=20` stops it splitting groups smaller than 20 patients. Without it the trees carry on splitting until each group is one person, which is memorising patients rather than learning patterns.

In [ ]:
from sklearn.impute import SimpleImputer

# Median learned from training data only, then applied to all three splits, so
# validation and test stay data the model has never learned anything from.
imputer = SimpleImputer(strategy="median")
X_train_filled = pd.DataFrame(imputer.fit_transform(X_train), columns=FEATURES, index=X_train.index)
X_val_filled   = pd.DataFrame(imputer.transform(X_val),   columns=FEATURES, index=X_val.index)
X_test_filled  = pd.DataFrame(imputer.transform(X_test),  columns=FEATURES, index=X_test.index)

print(f"Gaps in training set before fill: {X_train.isna().sum().sum():,}")
print(f"Gaps in training set after fill:  {X_train_filled.isna().sum().sum():,}")

In [14]:
from sklearn.ensemble import RandomForestClassifier

# min_samples_leaf=20 stops the trees splitting down to single patients.
# No class_weight, since balancing inflates the predicted risks and we score
# by ranking anyway, not by a yes/no cutoff.
rf = RandomForestClassifier(min_samples_leaf=20, random_state=42)
rf.fit(X_train_filled, y_train)

print(f"Random Forest trained on {len(X_train_filled):,} patients, {len(FEATURES)} features.")

Random Forest trained on 1,600 patients, 57 features.


## Step 5b: Add Gradient Boosting

A second model, so there's something to compare Random Forest against. Same data, same
split, same filled-in gaps. The only thing that changes is the model itself.

Where a Random Forest builds lots of trees independently and averages them, Gradient
Boosting builds them one after another, each one trying to fix the mistakes of the last.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

# min_samples_leaf=20 mirrors the forest, and is the minimum needed to stop this one
# memorising. Learning rate and depth are deliberately left alone until we can pick
# them against real validation data rather than synthetic noise.
gb = GradientBoostingClassifier(min_samples_leaf=20, random_state=42)
gb.fit(X_train_filled, y_train)

print(f"Gradient Boosting trained on {len(X_train_filled):,} patients, {len(FEATURES)} features.")

## Step 5c: Add Logistic Regression

This is a simple baseline model.

Why its done: Trees don't care what scale a number is on, but Logistic
Regression does, so a big number like glucose would drown out a small one like pH.
We use scaling as it puts every feature on the same level first,
learned from the training set only.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Put every feature on the same scale, learned from training data only.
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_filled), columns=FEATURES, index=X_train_filled.index)
X_val_scaled   = pd.DataFrame(scaler.transform(X_val_filled),   columns=FEATURES, index=X_val_filled.index)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test_filled),  columns=FEATURES, index=X_test_filled.index)

# max_iter raised from the default 100. No class_weight, same reason as the forest.
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)

print(f"Logistic Regression trained on {len(X_train_scaled):,} patients, {len(FEATURES)} features.")

## Step 6: Evaluate

Score all three models on the test set they've never seen, using three measures:

- **AUROC**: can it put the patients who died above the ones who didn't? 0.5 determines guess, 1.0 is perfect.
- **AUPRC**: Determines how good its high-risk flags specifically:
  because deaths are rare. A bad model scores 0.15 (the death rate itself.)
- **Brier**: are the predicted percentages honest? Lower is better, 0 is perfect.



In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

# Trees read the filled data, Logistic Regression reads the scaled version. Keeping
# that pairing in one place stops it being copy-pasted into every check below.
MODELS = [("Random Forest", rf, "filled"),
          ("Gradient Boosting", gb, "filled"),
          ("Logistic Regression", lr, "scaled")]

SPLITS = {"train": (X_train_filled, X_train_scaled, y_train),
          "val":   (X_val_filled,   X_val_scaled,   y_val),
          "test":  (X_test_filled,  X_test_scaled,  y_test)}

def predictions(split):
    """Each model's predicted death probability on the chosen split."""
    filled, scaled, y_true = SPLITS[split]
    return [(name, model.predict_proba(filled if kind == "filled" else scaled)[:, 1], y_true)
            for name, model, kind in MODELS]

def evaluate(y_true, probs):
    """The same three scores everywhere, so models and splits stay comparable."""
    return {"AUROC": roc_auc_score(y_true, probs),
            "AUPRC": average_precision_score(y_true, probs),
            "Brier": brier_score_loss(y_true, probs)}

In [ ]:
results = pd.DataFrame(
    [{"Model": name, **evaluate(y_true, probs)} for name, probs, y_true in predictions("val")]
).set_index("Model").round(3)

print(f"Validation set: {len(y_val):,} rows, {y_val.sum():,} deaths. Test stays sealed.")
print(results.to_string())

## Extra checks

Four quick checks on the models we already trained. Nothing new gets trained here.

The first two check the models themselves (are the percentages honest, is it memorising),
so they stay valid even on synthetic data. The last two describe how well it performs,
so on synthetic data they show the shape of the trade-off, not a real result.

In [ ]:
# Check 1: are the predicted percentages honest?
# Group patients by predicted risk, then see what fraction of each group actually died.
# If the model is honest, a "20 to 30%" group really should contain about 25% deaths.

bands = [0, 0.10, 0.20, 0.30, 0.40, 1.01]

for name, probs, y_true in predictions("val"):
    grouped = pd.DataFrame({"band": pd.cut(probs, bands, right=False), "died": y_true.values})
    table = grouped.groupby("band", observed=True).agg(
        patients=("died", "size"),
        actually_died=("died", "mean"),
    )
    shown = table.assign(
        actually_died=(table["actually_died"] * 100).round().astype(int).astype(str) + "%"
    )
    print(name)
    print(shown.to_string())

    # Honest means each band's real death rate climbs with the predicted one.
    if table["actually_died"].is_monotonic_increasing:
        print("  So: the real death rate climbs in step with the predicted one, so these")
        print("  percentages can be taken at face value.")
    else:
        print("  So: the real death rate jumps about instead of climbing steadily. This one")
        print("  can still rank patients, but poor percentage reliability.")
    print()

In [ ]:
# Check 2: is the model memorising instead of learning?
# Score it on the data it trained on, then on data it has never seen. A big gap means
# it memorised patients rather than finding patterns that carry over.

train_probs = {name: probs for name, probs, _ in predictions("train")}
val_probs = {name: probs for name, probs, _ in predictions("val")}

gaps = {}
for name in train_probs:
    train_score = roc_auc_score(y_train, train_probs[name])
    val_score = roc_auc_score(y_val, val_probs[name])
    gaps[name] = train_score - val_score
    print(f"{name:<20} train {train_score:.3f}   validation {val_score:.3f}   gap {gaps[name]:.3f}")

worst = max(gaps, key=gaps.get)
best = min(gaps, key=gaps.get)
print()
print(f"So: {worst} is the worst offender, scoring {gaps[worst]:.2f} higher on patients it")
print(f"has already seen. {best} has the smallest gap at {gaps[best]:.2f}, so it generalises best.")

In [ ]:
# Check 3: the precision and recall trade-off.
# Precision: of the patients we flag, how many really died.
# Recall:    of the patients who really died, how many we flagged.
# Moving the threshold trades one against the other, you can't have both.

import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, precision_score, recall_score

plt.figure(figsize=(7, 5))
for name, probs, y_true in predictions("val"):
    precision, recall, _ = precision_recall_curve(y_true, probs)
    plt.plot(recall, precision, label=name)

# A model with no skill would sit flat at the death rate.
plt.axhline(y_val.mean(), linestyle="--", color="grey", label="No skill (death rate)")
plt.xlabel("Recall: share of real deaths we caught")
plt.ylabel("Precision: share of our flags that were right")
plt.title("What you give up to catch more deaths")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print("So: the higher a line sits, the better that model is. The grey line is what you would")
print("get flagging patients at random, so anything near it is no better than guessing.")

In [ ]:
# Same trade-off as numbers, for the Random Forest.
rf_probs = val_probs["Random Forest"]

rows = []
for t in [0.10, 0.15, 0.20, 0.25, 0.30]:
    flagged = rf_probs >= t
    rows.append({
        "Threshold": t,
        "Patients flagged": flagged.sum(),
        "Precision": round(precision_score(y_val, flagged, zero_division=0), 2),
        "Recall": round(recall_score(y_val, flagged), 2),
    })

table = pd.DataFrame(rows).set_index("Threshold")
print(f"Random Forest, validation set: {len(y_val):,} patients, {y_val.sum():,} deaths")
print(table.to_string())

low, high = table.loc[0.10], table.loc[0.30]
print()
print(f"So: at 0.10 you catch {low['Recall']:.0%} of deaths but flag {low['Patients flagged']:.0f} of {len(y_val)} patients,")
print(f"which is most of the ward. At 0.30 only {high['Patients flagged']:.0f} get flagged, but you miss")
print(f"{1 - high['Recall']:.0%} of the deaths. Pick your poison.")

In [ ]:
# Check 4: what mistakes does each model actually make?
from sklearn.metrics import confusion_matrix

# 0.5 is no use here. With honest probabilities almost nobody crosses it, so it would
# flag nobody at all. The training death rate is the simplest defensible cutoff, and
# it is computed rather than hardcoded so it follows the data on the real BOLD swap.
# For triage, missing a death is worse than a false alarm, so we lean that way.
THRESHOLD = round(y_train.mean(), 2)
print(f"Threshold {THRESHOLD}, the training death rate. Chosen to catch most deaths,")
print("accepting more false alarms, which is the safer error in triage.")
print()

for name, probs, y_true in predictions("val"):
    flagged = probs >= THRESHOLD
    cleared_ok, false_alarms, missed, caught = confusion_matrix(y_true, flagged).ravel()
    print(name)
    print(f"  Deaths caught:            {caught:>4}  of {caught + missed}")
    print(f"  Deaths missed:            {missed:>4}")
    print(f"  False alarms:             {false_alarms:>4}  flagged but survived")
    print(f"  Correctly cleared:        {cleared_ok:>4}")
    print(f"  So: it catches {caught / (caught + missed):.0%} of the deaths, but for every one it correctly flags, it wrongly flags")
    print(f"  {false_alarms / caught:.1f} times more.")
    print()

## Step 7: Which biomarkers drive the prediction

Three separate ways of asking the same question, so we can see where they agree.

- **SHAP** for the two tree models. For each patient it works out how much each value
  pushed their risk up or down, then we average the size of those pushes.
- **Coefficients** from the Logistic Regression, a simpler and completely different sum.



In [ ]:
import shap

# TreeExplainer is the exact method for tree models. Run on validation, so the test
# set stays untouched until the end.
rf_explainer = shap.TreeExplainer(rf)
rf_shap = rf_explainer.shap_values(X_val_filled)

# For a 0/1 outcome, newer SHAP returns one set of values per class. Keep the "died" one.
if rf_shap.ndim == 3:
    rf_shap = rf_shap[:, :, 1]

# Average size of each feature's push, ignoring direction.
rf_ranking = pd.Series(abs(rf_shap).mean(axis=0), index=FEATURES).sort_values(ascending=False)

rf_ranking.head(15)[::-1].plot.barh(figsize=(7, 6), color="steelblue")
plt.xlabel("Average push on a patient's predicted risk")
plt.title("Random Forest: markers that move the prediction most")
plt.tight_layout()
plt.show()

top = rf_ranking.index[0]
print(f"So: {top} moves the forest's prediction more than anything else, by an average")
print(f"of {rf_ranking.iloc[0]:.4f}. The bars only show size, not whether a marker raises or lowers risk.")

In [ ]:
# Same again for Gradient Boosting. Two models trained differently, so if they agree
# on a marker that is more convincing than either on its own.
gb_shap = shap.TreeExplainer(gb).shap_values(X_val_filled)
if gb_shap.ndim == 3:
    gb_shap = gb_shap[:, :, 1]

gb_ranking = pd.Series(abs(gb_shap).mean(axis=0), index=FEATURES).sort_values(ascending=False)

gb_ranking.head(15)[::-1].plot.barh(figsize=(7, 6), color="darkorange")
plt.xlabel("Average push on a patient's predicted risk")
plt.title("Gradient Boosting: markers that move the prediction most")
plt.tight_layout()
plt.show()

shared = len(set(rf_ranking.head(15).index) & set(gb_ranking.head(15).index))
print(f"So: {gb_ranking.index[0]} comes top for this model. {shared} of the top 15 markers")
print("appear in both models' lists, so that much of the ranking is not model specific.")

In [25]:
# Line the three lists up side by side. Logistic Regression has no SHAP values, so we
# use the size of its coefficients instead, which is its own measure of importance.
lr_ranking = pd.Series(abs(lr.coef_[0]), index=FEATURES).sort_values(ascending=False)

comparison = pd.DataFrame({
    "RF (SHAP)": rf_ranking.head(10).index,
    "GB (SHAP)": gb_ranking.head(10).index,
    "LR (coefficient)": lr_ranking.head(10).index,
})
comparison.index = range(1, 11)
print("Top 10 markers by each method:")
print(comparison.to_string())

# Markers all three methods put in their top 10.
agreed = set(rf_ranking.head(10).index) & set(gb_ranking.head(10).index) & set(lr_ranking.head(10).index)
print()
if agreed:
    print(f"So: {len(agreed)} marker(s) appear in all three top tens: {', '.join(sorted(agreed))}.")
    print("Those are the ones that do not depend on which model you picked.")
else:
    print("So: no marker makes all three top tens, which means the ranking depends heavily")
    print("on the model. Treat any single list with caution.")

Top 10 markers by each method:
                        RF (SHAP)               GB (SHAP)               LR (coefficient)
1          sofa_past_overall_24hr  sofa_past_overall_24hr                  admission_age
2                   admission_age           admission_age         sofa_past_overall_24hr
3                     bmp_lactate          bmp_creatinine                    bmp_lactate
4      sofa_past_coagulation_24hr             bmp_lactate             sofa_past_cns_24hr
5                  bmp_creatinine                coag_inr  sofa_past_cardiovascular_24hr
6              sofa_past_cns_24hr            cbc_platelet     sofa_past_coagulation_24hr
7            sofa_past_liver_24hr                    SaO2                 bmp_creatinine
8   sofa_past_cardiovascular_24hr                 cbc_rdw                       coag_inr
9            sofa_past_renal_24hr       vitals_heart_rate                  BMI_admission
10                       coag_inr                      pH           sofa_past_l

In [ ]:
# One patient, explained. This is the part that matters clinically: not just which
# markers matter in general, but why this person got the score they did.
patient = rf_probs.argmax()

pushes = pd.Series(rf_shap[patient], index=FEATURES).sort_values(key=abs, ascending=False).head(8)
breakdown = pd.DataFrame({
    "Their value": X_val_filled.iloc[patient][pushes.index].round(2),
    "Push on risk": pushes.round(4),
})

print(f"Highest risk patient in the validation set: {rf_probs[patient]:.0%} predicted, actually "
      f"{'died' if y_val.iloc[patient] == 1 else 'survived'}.")
print(f"Average patient is {rf_probs.mean():.0%}.")
print()
print(breakdown.to_string())

raised = pushes[pushes > 0]
print()
print(f"So: {raised.index[0]} pushed this patient's risk up the most. Positive numbers raise")
print("the risk, negative ones lower it, and they add up to the gap from the average patient.")

## Step 8: Do the top markers hold up on their own?

A marker can look important just because sicker, older patients tend to have worse
readings and die more, not because it actually matters.

So each marker is tested twice: alone, then again alongside age and severity. If its
effect mostly disappears the second time, it wasn't the marker driving things, it was
age or severity.

Just a confounder check, not full causal inference.

In [27]:
import numpy as np
import statsmodels.api as sm

# The two things we suspect are behind everything else.
CONFOUNDERS = ["admission_age", "sofa_past_overall_24hr"]

# Markers all three methods agreed on in Step 7, minus the confounders themselves.
to_check = [m for m in sorted(agreed) if m not in CONFOUNDERS]

rows = []
for marker in to_check:
    # Same standardised training data as the model, so an odds ratio reads
    # "per 1 standard deviation increase" and markers compare like for like.
    alone = sm.Logit(y_train, sm.add_constant(X_train_scaled[[marker]])).fit(disp=0)
    adjusted = sm.Logit(y_train, sm.add_constant(X_train_scaled[[marker] + CONFOUNDERS])).fit(disp=0)

    before, after = alone.params[marker], adjusted.params[marker]
    p_after = adjusted.pvalues[marker]

    # How much of the effect survived once age and severity were in the room.
    kept = abs(after) / abs(before) if before != 0 else 0
    if p_after >= 0.05:
        verdict = "explained away"
    elif kept < 0.5:
        verdict = "weakened"
    else:
        verdict = "holds up"

    rows.append({
        "Marker": marker,
        "Odds ratio alone": round(np.exp(before), 2),
        "Odds ratio adjusted": round(np.exp(after), 2),
        "p-value": round(p_after, 3),
        "Verdict": verdict,
    })

results_8 = pd.DataFrame(rows).set_index("Marker")
print(f"Adjusted for: {', '.join(CONFOUNDERS)}")
print("Odds ratio is per 1 standard deviation. Above 1 means higher risk, 1.0 means no effect.")
print()
print(results_8.to_string())

held = results_8[results_8["Verdict"] == "holds up"]
print()
if len(held):
    print(f"So: {', '.join(held.index)} still carries weight after age and severity are")
    print("accounted for, so it is not simply a stand-in for how old or how sick the patient is.")
else:
    print("So: none of these markers survive once age and severity are accounted for. On this")
    print("data they look like proxies for how old and how sick the patient already was.")

Adjusted for: admission_age, sofa_past_overall_24hr
Odds ratio is per 1 standard deviation. Above 1 means higher risk, 1.0 means no effect.

                Odds ratio alone  Odds ratio adjusted  p-value   Verdict
Marker                                                                  
bmp_creatinine              1.20                 1.26    0.001  holds up
bmp_lactate                 1.43                 1.49    0.000  holds up
coag_inr                    1.16                 1.21    0.009  holds up

So: bmp_creatinine, bmp_lactate, coag_inr still carries weight after age and severity are
accounted for, so it is not simply a stand-in for how old or how sick the patient is.


## Final: the test set, opened once

Everything above reads the validation split. This is the only cell that touches test,
and it runs after the tuning decisions are already made, so these numbers are not
something we have quietly optimised towards.

In [ ]:
# Test set, read once. Nothing gets tuned on the back of these numbers.
final = pd.DataFrame(
    [{"Model": name, **evaluate(y_true, probs)} for name, probs, y_true in predictions("test")]
).set_index("Model").round(3)

print(f"Test set: {len(y_test):,} rows, {y_test.sum():,} deaths, {y_test.mean() * 100:.1f}% died")
print(final.to_string())

best_model = final["AUROC"].idxmax()
print()
print(f"So: {best_model} ranks unseen patients best, at AUROC {final.loc[best_model, 'AUROC']:.3f},")
print(f"against 0.50 for a coin flip. AUPRC has to beat the death rate of {y_test.mean():.3f}")
print("to be worth anything, since that is what flagging at random would score.")